# SemEval-2026 Task 13 — Subtask B: Improved Model

**Multi-Class Code Authorship Detection** (11 classes: human + 10 LLM families)

### Workflow:
1. Run setup cells (install, data, imports, class definitions)
2. **Step 1:** HP search (~50 min) — finds best learning rate + max_length
3. **Step 2:** Final training with best params (~30-40 min)
4. **Step 3:** Generate predictions on test set

**Runtime:** Set to `T4 GPU` via `Runtime → Change runtime type`

In [27]:
!pip install --upgrade transformers datasets scikit-learn accelerate -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Data Setup
Uncomment the option matching your environment.

In [28]:
# ============================================================
# OPTION A: Clone repo (for Google Colab)
# ============================================================
# !git clone https://github.com/mbzuai-nlp/SemEval-2026-Task13.git
# TRAIN_PATH = "SemEval-2026-Task13/task_b/task_b_training_set.parquet"
# VAL_PATH   = "SemEval-2026-Task13/task_b/task_b_validation_set.parquet"
# TEST_PATH  = "SemEval-2026-Task13/task_b/task_b_test_set_sample.parquet"

# ============================================================
# OPTION B: Kaggle input paths
# ============================================================
# TRAIN_PATH = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_training_set.parquet"
# VAL_PATH   = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_validation_set.parquet"
# TEST_PATH  = "/kaggle/input/datasets/daniilor/semeval-2026-task13/SemEval-2026-Task13/task_b/task_b_test_set_sample.parquet"

# ============================================================
# OPTION C: Download from HuggingFace
# ============================================================
from datasets import load_dataset
print("Downloading SemEval-2026-Task13 Subtask B from HuggingFace...")
hf_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "B")
hf_dataset['train'].to_parquet('task_b_train.parquet')
hf_dataset['validation'].to_parquet('task_b_val.parquet')
if 'test' in hf_dataset:
    hf_dataset['test'].to_parquet('task_b_test.parquet')
    TEST_PATH = 'task_b_test.parquet'
else:
    TEST_PATH = None
    print("No test split available on HuggingFace.")
TRAIN_PATH = 'task_b_train.parquet'
VAL_PATH   = 'task_b_val.parquet'
print("Done!")

# ============================================================
print(f"Train: {TRAIN_PATH}")
print(f"Val:   {VAL_PATH}")
print(f"Test:  {TEST_PATH}")

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Done!
Train: task_b_train.parquet
Val:   task_b_val.parquet
Test:  task_b_test.parquet


## Configuration

In [31]:
USE_SUBSET = True            # False = full 500K dataset (slower)
HUMAN_SUBSET_SIZE = 40000    # Increased from 20K — more human examples reduce overfitting
VAL_FRACTION = 0.2           # Keep 20% of validation

In [32]:
import os
os.environ["WANDB_DISABLED"] = "true"
import gc
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import Dataset
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score
)
import warnings
warnings.filterwarnings("ignore")

# Check transformers version for API compatibility
TRANSFORMERS_VERSION = tuple(int(x) for x in transformers.__version__.split('.')[:2])
print(f"Transformers version: {transformers.__version__}")

ID_TO_LABEL = {
    0: "human", 1: "deepseek", 2: "qwen", 3: "01-ai", 4: "bigcode",
    5: "gemma", 6: "phi", 7: "meta-llama", 8: "ibm-granite", 9: "mistral", 10: "openai"
}
LABEL_TO_ID = {v: k for k, v in ID_TO_LABEL.items()}
NUM_LABELS = len(ID_TO_LABEL)

print(f"Task B: {NUM_LABELS}-class classification")
print(f"Labels: {list(ID_TO_LABEL.values())}")
print(f"Subset mode: {USE_SUBSET}")

Transformers version: 4.45.0
Task B: 11-class classification
Labels: ['human', 'deepseek', 'qwen', '01-ai', 'bigcode', 'gemma', 'phi', 'meta-llama', 'ibm-granite', 'mistral', 'openai']
Subset mode: True


## Model & Training Classes

In [33]:
class WeightedTrainer(Trainer):
    """
    Custom Trainer with class-weighted CrossEntropyLoss.
    Handles label_smoothing from TrainingArguments + class weights.
    """

    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        if class_weights is not None:
            self.class_weights = class_weights.to(self.model.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        smoothing = self.args.label_smoothing_factor if self.args.label_smoothing_factor else 0.0

        if self.class_weights is not None:
            weights = self.class_weights.to(logits.device)
            loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=smoothing)
        else:
            loss_fn = nn.CrossEntropyLoss(label_smoothing=smoothing)

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [34]:
def stratified_subsample(df, human_subset_size=20000, random_state=42):
    """Keep ALL minority samples, downsample human class."""
    human_df = df[df['label'] == 0]
    minority_df = df[df['label'] != 0]

    if len(human_df) > human_subset_size:
        human_df = human_df.sample(n=human_subset_size, random_state=random_state)

    result = pd.concat([human_df, minority_df], ignore_index=True)
    result = result.sample(frac=1, random_state=random_state).reset_index(drop=True)

    print(f"  Subsampled: {len(df)} -> {len(result)} samples")
    print(f"  Human: {len(human_df)} | Minority (all 10 classes): {len(minority_df)}")
    return result


def compute_class_weights(label_counts, num_labels, method="sqrt"):
    """
    Compute class weights with sqrt scaling to prevent extreme ratios.
    Raw inverse frequency: 220:1 ratio -> unstable.
    sqrt scaling: ~15:1 ratio -> stable.
    """
    total = sum(label_counts.get(i, 1) for i in range(num_labels))
    weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)
        raw_weight = total / (num_labels * count)
        if method == "sqrt":
            w = np.sqrt(raw_weight)
        elif method == "log":
            w = np.log1p(raw_weight)
        else:
            w = raw_weight
        weights.append(w)

    min_w = min(weights)
    weights = [w / min_w for w in weights]
    return torch.FloatTensor(weights)

In [42]:
class ImprovedCodeTrainer:
    def __init__(self, max_length=512, model_name="microsoft/unixcoder-base"):
        self.max_length = max_length
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.class_weights = None

    def load_and_prepare_data(self):
        try:
            df = pd.read_parquet(TRAIN_PATH)
            print(f"Dataset columns: {df.columns.tolist()}")
            print(f"Sample data:\n{df.head()}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            unique_labels = sorted(df['label'].unique())
            print(f"\nUnique labels: {unique_labels}")

            print(f"\nFull dataset label distribution:")
            label_counts = df['label'].value_counts().sort_index()
            for label_id, count in label_counts.items():
                name = ID_TO_LABEL.get(label_id, f"unknown-{label_id}")
                print(f"  {label_id} ({name:>12s}): {count:>7d} ({count/len(df)*100:.1f}%)")

            if USE_SUBSET:
                print(f"\n--- SUBSET MODE ---")
                df = stratified_subsample(df, human_subset_size=HUMAN_SUBSET_SIZE)

            label_counts = df['label'].value_counts().sort_index()
            self.class_weights = compute_class_weights(label_counts, NUM_LABELS, method="sqrt")

            print(f"\nClass weights (sqrt-scaled):")
            for i, w in enumerate(self.class_weights.tolist()):
                name = ID_TO_LABEL.get(i, f"unknown-{i}")
                count = label_counts.get(i, 0)
                print(f"  {name:>12s}: weight={w:>5.2f}  (n={count})")

            val_df = pd.read_parquet(VAL_PATH)
            val_df['label'] = val_df['label'].astype(int)

            if USE_SUBSET:
                print(f"\n--- Subsampling validation to {VAL_FRACTION*100:.0f}% ---")
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(frac=VAL_FRACTION, random_state=42)
                ).reset_index(drop=True)

            print(f"\nFinal -> Train: {len(df)} | Val: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        print(f"\nInitializing {self.model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=NUM_LABELS,
            problem_type="single_label_classification"
        ).to('cuda')
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"Model loaded: {total_params/1e6:.1f}M params")

    # In the ImprovedCodeTrainer class, replace tokenize_function:
    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            max_length=self.max_length,
            # NO padding here — DataCollatorWithPadding handles it per batch
            # NO return_tensors — datasets.map() expects lists, not tensors
        )

    def prepare_datasets(self, train_df, val_df):
        print("\nTokenizing datasets...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset = Dataset.from_pandas(val_df[['code', 'label']])

        train_dataset = train_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])
        val_dataset = val_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset = val_dataset.rename_column('label', 'labels')

        print(f"Done. Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        preds = np.argmax(predictions, axis=1)

        unique, counts = np.unique(preds, return_counts=True)
        pred_dist = {int(u): int(c) for u, c in zip(unique, counts)}
        print(f"  Prediction distribution: {pred_dist}")

        accuracy = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
        weighted_f1 = f1_score(labels, preds, average='weighted', zero_division=0)

        return {
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'accuracy': accuracy,
        }

    # In the ImprovedCodeTrainer class, replace the train() method:
    def train(self, train_dataset, val_dataset, output_dir="./results",
              num_epochs=3, batch_size=16, learning_rate=2e-5):
        print("\n" + "="*60)
        print("STARTING TRAINING")
        print("="*60)

        steps_per_epoch = len(train_dataset) // (batch_size * 2)
        eval_save_steps = max(100, steps_per_epoch // 2)  # Evaluate 2x per epoch (less overhead)

        # Build TrainingArguments with version-compatible parameter names
        train_args_dict = dict(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            gradient_accumulation_steps=2,
            warmup_ratio=0.1,
            weight_decay=0.05,                # Increased from 0.01 to combat overfitting
            logging_steps=50,
            save_strategy="steps",
            save_steps=eval_save_steps,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine",
            save_total_limit=1,               # Only keep best checkpoint (saves disk)
            fp16=True,
            report_to="none",
            label_smoothing_factor=0.1,       # Reduces overconfidence, helps minority classes
        )

        # Handle eval_strategy vs evaluation_strategy across versions
        try:
            train_args_dict['eval_strategy'] = 'steps'
            train_args_dict['eval_steps'] = eval_save_steps
            training_args = TrainingArguments(**train_args_dict)
        except TypeError:
            del train_args_dict['eval_strategy']
            train_args_dict['evaluation_strategy'] = 'steps'
            train_args_dict['eval_steps'] = eval_save_steps
            training_args = TrainingArguments(**train_args_dict)

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        # Handle tokenizer vs processing_class across versions
        trainer_kwargs = dict(
            class_weights=self.class_weights,
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )
        try:
            trainer = WeightedTrainer(processing_class=self.tokenizer, **trainer_kwargs)
        except TypeError:
            trainer = WeightedTrainer(tokenizer=self.tokenizer, **trainer_kwargs)

        print(f"Model: {self.model_name}")
        print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
        print(f"Batch: {batch_size}x2, LR: {learning_rate}, MaxLen: {self.max_length}")
        print(f"Epochs: {num_epochs}, Eval every {eval_save_steps} steps")
        print(f"fp16: True, Scheduler: cosine, Label smoothing: 0.1")
        print(f"Weight decay: 0.05, Save limit: 1")
        print()

        trainer.train()
        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"\nModel saved to {output_dir}")
        return trainer

    def evaluate_model(self, trainer, val_dataset):
        print("\n" + "="*60)
        print("EVALUATION")
        print("="*60)
        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        unique, counts = np.unique(y_pred, return_counts=True)
        print("\nPrediction distribution:")
        for u, c in zip(unique, counts):
            name = ID_TO_LABEL.get(int(u), f"unknown-{u}")
            print(f"  Predicted {name:>12s} (label {u}): {c} times")

        target_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
        print("\nPer-Class Classification Report:")
        print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
        print(f"\n>>> COMPETITION METRIC (Macro F1): {macro_f1:.4f} <<<")
        return predictions

    def run_full_pipeline(self, output_dir="./results", num_epochs=3,
                          batch_size=16, learning_rate=2e-5):
        try:
            train_df, val_df = self.load_and_prepare_data()
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate
            )
            self.evaluate_model(trainer, val_dataset)
            print("\nPipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"Error in pipeline: {e}")
            raise

---
## Step 1: Quick HP Search (~50 min on T4)

Tests 6 combos: learning_rate × max_length, 2 epochs each.

**Skip this cell** if you want to go straight to training with defaults.

In [36]:
    SEARCH_GRID = {
    'learning_rate': [1e-5, 2e-5, 5e-5],
    'max_length': [128, 256],
}
SEARCH_EPOCHS = 2

results = []

for lr in SEARCH_GRID['learning_rate']:
    for ml in SEARCH_GRID['max_length']:
        print(f"\n{'#'*60}")
        print(f"TRIAL: lr={lr}, max_length={ml}")
        print(f"{'#'*60}")

        trial_obj = ImprovedCodeTrainer(
            max_length=ml,
            model_name="microsoft/unixcoder-base",
        )

        try:
            trial_trainer = trial_obj.run_full_pipeline(
                output_dir=f"hp_search_lr{lr}_ml{ml}",
                num_epochs=SEARCH_EPOCHS,
                batch_size=16,
                learning_rate=lr
            )

            eval_results = [x for x in trial_trainer.state.log_history if 'eval_macro_f1' in x]
            best_f1 = max([x['eval_macro_f1'] for x in eval_results]) if eval_results else 0.0

            results.append({'learning_rate': lr, 'max_length': ml, 'best_macro_f1': best_f1})
            print(f"\n>>> Trial result: Macro F1 = {best_f1:.4f}")

        except Exception as e:
            print(f"Trial failed: {e}")
            results.append({'learning_rate': lr, 'max_length': ml, 'best_macro_f1': 0.0})

        del trial_obj
        try: del trial_trainer
        except: pass
        gc.collect()
        torch.cuda.empty_cache()

print("\n" + "="*60)
print("HYPERPARAMETER SEARCH RESULTS")
print("="*60)
results_df = pd.DataFrame(results).sort_values('best_macro_f1', ascending=False)
print(results_df.to_string(index=False))

best = results_df.iloc[0]
BEST_LR = best['learning_rate']
BEST_ML = int(best['max_length'])
print(f"\n>>> BEST: lr={BEST_LR}, max_length={BEST_ML}, Macro F1={best['best_macro_f1']:.4f}")


############################################################
TRIAL: lr=1e-05, max_length=128
############################################################
Dataset columns: ['code', 'generator', 'label', 'language']
Sample data:
                                                code generator  label language
0  def load(config, filepath, token):\n    if con...     Human      0   Python
1  n = int(input())\narr = list(map(int, input()....     Human      0   Python
2  using Aow.Infrastructure.Domain;\nusing Aow.In...    GPT-4o     10       C#
3  def save_data(bot, force=False):\n    if bot.d...     Human      0   Python
4  def parse_metadata(metaurl, progress=1e5):\n  ...     Human      0   Python

Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

Full dataset label distribution:
  0 (       human):  442096 (88.4%)
  1 (    deepseek):    4162 (0.8%)
  2 (        qwen):    8993 (1.8

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: 125.9M params

Tokenizing datasets...


Map:   0%|          | 0/77904 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/77904 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Done. Train: 77904, Val: 20000

STARTING TRAINING
Model: microsoft/unixcoder-base
Train: 77904, Val: 20000
Batch: 16x2, LR: 5e-05, MaxLen: 256
Epochs: 2, Eval every 811 steps
fp16: True, Scheduler: cosine



Step,Training Loss,Validation Loss,Macro F1,Weighted F1,Accuracy
811,1.307600,0.558454,0.377681,0.863325,0.827250
1622,1.014000,0.369342,0.453376,0.899124,0.880700


  Prediction distribution: {0: 15420, 1: 1113, 2: 488, 3: 181, 4: 245, 5: 306, 6: 242, 7: 447, 8: 441, 9: 208, 10: 909}
  Prediction distribution: {0: 16398, 1: 620, 2: 315, 3: 294, 4: 194, 5: 107, 6: 238, 7: 526, 8: 424, 9: 214, 10: 670}
Error in pipeline: [enforce fail at inline_container.cc:664] . unexpected pos 850823168 vs 850823060
Trial failed: [enforce fail at inline_container.cc:664] . unexpected pos 850823168 vs 850823060

HYPERPARAMETER SEARCH RESULTS
 learning_rate  max_length  best_macro_f1
       0.00002         256       0.415380
       0.00005         128       0.388635
       0.00002         128       0.361569
       0.00001         256       0.355514
       0.00001         128       0.319913
       0.00005         256       0.000000

>>> BEST: lr=2e-05, max_length=256, Macro F1=0.4154


---
## Step 2: Final Training

If you skipped HP search, uncomment the manual lines below.

In [ ]:
# If you skipped HP search, set manually:
# BEST_LR = 2e-5
# BEST_ML = 256

import shutil, glob

# Clean up HP search checkpoints to free disk space
for d in glob.glob("hp_search_*"):
    shutil.rmtree(d, ignore_errors=True)
    print(f"Removed: {d}")

# Also clean any previous results
for d in glob.glob("results*") + glob.glob("taskB-improved-model*") + glob.glob("/kaggle/working/taskB*"):
    shutil.rmtree(d, ignore_errors=True)
    print(f"Removed: {d}")

OUTPUT_DIR = "/kaggle/working/taskB-improved-model"

gc.collect()
torch.cuda.empty_cache()

trainer_obj = ImprovedCodeTrainer(
    max_length=BEST_ML,
    model_name="microsoft/unixcoder-base",
)

trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=5,        # Reduced: overfitting after epoch 5
    batch_size=16,
    learning_rate=BEST_LR
)

Removed: hp_search_lr2e-05_ml256
Removed: hp_search_lr5e-05_ml256
Removed: hp_search_lr5e-05_ml128
Removed: hp_search_lr1e-05_ml256
Removed: hp_search_lr2e-05_ml128
Removed: hp_search_lr1e-05_ml128
Dataset columns: ['code', 'generator', 'label', 'language']
Sample data:
                                                code generator  label language
0  def load(config, filepath, token):\n    if con...     Human      0   Python
1  n = int(input())\narr = list(map(int, input()....     Human      0   Python
2  using Aow.Infrastructure.Domain;\nusing Aow.In...    GPT-4o     10       C#
3  def save_data(bot, force=False):\n    if bot.d...     Human      0   Python
4  def parse_metadata(metaurl, progress=1e5):\n  ...     Human      0   Python

Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

Full dataset label distribution:
  0 (       human):  442096 (88.4%)
  1 (    deepseek):    4

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: 125.9M params

Tokenizing datasets...


Map:   0%|          | 0/77904 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Done. Train: 77904, Val: 20000

STARTING TRAINING
Model: microsoft/unixcoder-base
Train: 77904, Val: 20000
Batch: 16x2, LR: 2e-05, MaxLen: 256
Epochs: 10, Eval every 811 steps
fp16: True, Scheduler: cosine



Step,Training Loss,Validation Loss,Macro F1,Weighted F1,Accuracy
811,1.567300,0.495084,0.331747,0.877312,0.857850
1622,1.271900,0.500245,0.385151,0.873435,0.845300
2433,1.158200,0.373870,0.431992,0.896960,0.883550
3244,1.069500,0.408211,0.441687,0.893462,0.872950
4055,0.867700,0.442571,0.448543,0.888136,0.863550
4866,0.881800,0.432953,0.441923,0.890283,0.868150
5677,0.727800,0.414126,0.454769,0.897408,0.877900
6488,0.601400,0.449060,0.469530,0.896339,0.873850


  Prediction distribution: {0: 16358, 1: 515, 2: 271, 3: 95, 4: 313, 5: 337, 6: 210, 7: 226, 8: 469, 9: 195, 10: 1011}
  Prediction distribution: {0: 15797, 1: 713, 2: 251, 3: 398, 4: 241, 5: 121, 6: 224, 7: 664, 8: 467, 9: 146, 10: 978}
  Prediction distribution: {0: 16547, 1: 376, 2: 185, 3: 235, 4: 206, 5: 145, 6: 247, 7: 429, 8: 546, 9: 198, 10: 886}
  Prediction distribution: {0: 16271, 1: 374, 2: 525, 3: 369, 4: 203, 5: 99, 6: 202, 7: 368, 8: 632, 9: 341, 10: 616}
  Prediction distribution: {0: 16012, 1: 810, 2: 545, 3: 321, 4: 207, 5: 133, 6: 268, 7: 376, 8: 480, 9: 149, 10: 699}
  Prediction distribution: {0: 16146, 1: 609, 2: 417, 3: 386, 4: 184, 5: 145, 6: 165, 7: 351, 8: 477, 9: 318, 10: 802}
  Prediction distribution: {0: 16328, 1: 516, 2: 486, 3: 394, 4: 209, 5: 178, 6: 196, 7: 377, 8: 450, 9: 232, 10: 634}
  Prediction distribution: {0: 16232, 1: 630, 2: 541, 3: 202, 4: 130, 5: 183, 6: 210, 7: 461, 8: 409, 9: 427, 10: 575}


---
## Step 3: Predict on Test Set

In [38]:
import torch
from itertools import chain
from datasets import load_dataset
from tqdm import tqdm

@torch.no_grad()
def predict_with_trainer(trainer_obj, parquet_path, output_path,
                         max_length=512, batch_size=16, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = trainer_obj.model
    tokenizer = trainer_obj.tokenizer
    if tokenizer is None:
        raise ValueError("trainer_obj must have a tokenizer.")

    model.to(device)
    model.eval()

    ds = load_dataset("parquet", data_files=parquet_path, split="train", streaming=True)
    it = iter(ds)
    first = next(it)
    if not {"ID", "code"}.issubset(first.keys()):
        raise ValueError("Parquet must contain 'ID' and 'code' columns")
    stream = chain([first], it)

    def batcher(iterator, bs):
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    with open(output_path, "w") as f:
        f.write("ID,prediction\n")
        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            codes = [row["code"] for row in batch]
            ids   = [row["ID"] for row in batch]
            enc = tokenizer(codes, truncation=True, padding=True,
                           max_length=max_length, return_tensors="pt")
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()
            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    print(f"Predictions saved to {output_path}")

In [39]:
if TEST_PATH is not None:
    OUT_CSV = "submission_b.csv"
    predict_with_trainer(
        trainer_obj=trainer_obj,
        parquet_path=TEST_PATH,
        output_path=OUT_CSV,
        max_length=BEST_ML,
        batch_size=32,
        device="cuda"
    )
    print("Wrote:", OUT_CSV)
else:
    print("No test path set. Update TEST_PATH in the data setup cell.")

NameError: name 't' is not defined

In [ ]:
# Download submission (Colab only)
# from google.colab import files
# files.download(OUT_CSV)